# <center>Лабораторна робота № 1.<br> Аналіз даних про доходи населення</center>


**В завданні пропонується за допомогою Pandas відповісти на декілька питань за даними репозиторія UCI [Adult](https://archive.ics.uci.edu/ml/datasets/Adult) (качати дані не потрібно – вони вже є в директорії роботи). 

Унікальні значення ознак (більше інформації за посиланням вище):
- age: continuous.
- workclass: Private, Self-emp-not-inc, Self-emp-inc, Federal-gov, Local-gov, State-gov, Without-pay, Never-worked.
- fnlwgt: continuous.
- education: Bachelors, Some-college, 11th, HS-grad, Prof-school, Assoc-acdm, Assoc-voc, 9th, 7th-8th, 12th, Masters, 1st-4th, 10th, Doctorate, 5th-6th, Preschool.
- education-num: continuous.
- marital-status: Married-civ-spouse, Divorced, Never-married, Separated, Widowed, Married-spouse-absent, Married-AF-spouse.
- occupation: Tech-support, Craft-repair, Other-service, Sales, Exec-managerial, Prof-specialty, Handlers-cleaners, Machine-op-inspct, Adm-clerical, Farming-fishing, Transport-moving, Priv-house-serv, Protective-serv, Armed-Forces.
- relationship: Wife, Own-child, Husband, Not-in-family, Other-relative, Unmarried.
- race: White, Asian-Pac-Islander, Amer-Indian-Eskimo, Other, Black.
- sex: Female, Male.
- capital-gain: continuous.
- capital-loss: continuous.
- hours-per-week: continuous.
- native-country: United-States, Cambodia, England, Puerto-Rico, Canada, Germany, Outlying-US(Guam-USVI-etc), India, Japan, Greece, South, China, Cuba, Iran, Honduras, Philippines, Italy, Poland, Jamaica, Vietnam, Mexico, Portugal, Ireland, France, Dominican-Republic, Laos, Ecuador, Taiwan, Haiti, Columbia, Hungary, Guatemala, Nicaragua, Scotland, Thailand, Yugoslavia, El-Salvador, Trinadad&Tobago, Peru, Hong, Holand-Netherlands.   
- salary: >50K,<=50K

In [14]:
import pandas as pd

**Доступ до даних на google drive**, якщо ви відкриваєте блокнот в **google colab**, а не на PC, можна отримати шляхом монтування google drive

In [15]:
# Локальні дані знаходяться в lab1/data/.

In [16]:
from pathlib import Path

data_folder = Path("data")
if not (data_folder / "adult.data.csv").exists():
    data_folder = Path("lab1/data")

In [17]:
data = pd.read_csv(data_folder / "adult.data.csv")
data.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,salary
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


**1. Скільки чоловіків і жінок (ознака *sex*) представлено в цьому наборі даних?**

In [18]:
data["sex"].value_counts()

sex
Male      21790
Female    10771
Name: count, dtype: int64

**2. Який середній вік (ознака *age*) жінок?**

In [19]:
data.loc[data["sex"] == "Female", "age"].mean().round(2)

np.float64(36.86)

**3. Яка частка громадян Німеччини (ознака *native-country*)?**

In [20]:
germany_share = (data["native-country"] == "Germany").mean()
print(f"Частка громадян Німеччини: {germany_share:.2%}")

Частка громадян Німеччини: 0.42%


**4-5. Які середні значення і середні відхилення віку тих, хто отримує більше 50K в рік (ознака *salary*) і тих, хто отримує менше 50K в рік? **

In [21]:
salary_age_stats = data.groupby("salary")["age"].agg(["mean", "std"]).round(2)
salary_age_stats.rename(index={">50K": "більше 50K", "<=50K": "не більше 50K"})

,mean,std
salary,,
не більше 50K,36.78,14.02
більше 50K,44.25,10.52


**6. Чи правда, що люди, які отримують більше 50k, мають як мінімум вищу освіту? (ознака *education – Bachelors, Prof-school, Assoc-acdm, Assoc-voc, Masters* чи *Doctorate*)**

In [22]:
high_salary_education = {"Bachelors", "Prof-school", "Assoc-acdm", "Assoc-voc", "Masters", "Doctorate"}
high_salary = data["salary"] == ">50K"
is_higher_education = data.loc[high_salary, "education"].isin(high_salary_education)
print("Усі мають вищу освіту:", is_higher_education.all())
print(f"Частка з визначеною вищою освітою: {is_higher_education.mean():.2%}")

Усі мають вищу освіту: False
Частка з визначеною вищою освітою: 57.84%


**7. Вивести статистику віку для кажної раси (ознака *race*) і кожної статі. Використовуйте *groupby* і *describe*. Знайдіть таким чином максимальний вік чоловіків раси *Amer-Indian-Eskimo*.**

In [23]:
age_by_race_sex = data.groupby(["race", "sex"])["age"].describe().round(2)
display(age_by_race_sex)
max_age = data.loc[(data["race"] == "Amer-Indian-Eskimo") & (data["sex"] == "Male"), "age"].max()
print("Максимальний вік чоловіків Amer-Indian-Eskimo:", max_age)

count   mean    std   min   25%   50%    75%  \
race               sex                                                      
Amer-Indian-Eskimo Female    119.0  37.12  13.11  17.0  27.0  36.0  46.00   
                   Male      192.0  37.21  12.05  17.0  28.0  35.0  45.00   
Asian-Pac-Islander Female    346.0  35.09  12.30  17.0  25.0  33.0  43.75   
                   Male      693.0  39.07  12.88  18.0  29.0  37.0  46.00   
Black              Female   1555.0  37.85  12.64  17.0  28.0  37.0  46.00   
                   Male     1569.0  37.68  12.88  17.0  27.0  36.0  46.00   
Other              Female    109.0  31.68  11.63  17.0  23.0  29.0  39.00   
                   Male      162.0  34.65  11.36  17.0  26.0  32.0  42.00   
White              Female   8642.0  36.81  14.33  17.0  25.0  35.0  46.00   
                   Male    19174.0  39.65  13.44  17.0  29.0  38.0  49.00   

                            max  
race               sex           
Amer-Indian-Eskimo Female  80.0  
                   Male    82.0  
Asian-Pac-Islander Female  75.0  
                   Male    90.0  
Black              Female  90.0  
                   Male    90.0  
Other              Female  74.0  
                   Male    77.0  
White              Female  90.0  
                   Male    90.0

Максимальний вік чоловіків Amer-Indian-Eskimo: 82


**8. Серед кого більша частка заробляючих багато (>50K): серед одружених чи холостих чоловіків (ознака *marital-status*)? Одруженими вважаємо тих, у кого *marital-status* починається з *Married* (Married-civ-spouse, Married-spouse-absent чи Married-AF-spouse), решту вважаємо холостими.**

In [24]:
men = data["sex"] == "Male"
married = data["marital-status"].str.startswith("Married")
status = married[men].map({True: "одружені", False: "холості"})
high_salary_share = data.loc[men].assign(status=status).groupby("status")["salary"].apply(lambda s: (s == ">50K").mean())
(high_salary_share * 100).round(2).rename("частка >50K, %")

status
одружені    44.05
холості      8.45
Name: частка >50K, %, dtype: float64

**9. Яку максимальну кількість годин людина працює за тиждень (ознака *hours-per-week*)? Скільки людей працюють таку кількість годин і який серед них відсоток заробляючих багато?**

In [25]:
max_hours = data["hours-per-week"].max()
max_hours_workers = data[data["hours-per-week"] == max_hours]
high_salary_share = (max_hours_workers["salary"] == ">50K").mean()
print("Максимум годин на тиждень:", max_hours)
print("Кількість людей:", len(max_hours_workers))
print(f"Частка з доходом >50K: {high_salary_share:.2%}")

Максимум годин на тиждень: 99
Кількість людей: 85
Частка з доходом >50K: 29.41%


**10. Підрахуйте середній час роботи (*hours-per-week*) заробляючих мало і багато (*salary*) для кожної країни (*native-country*).**

In [26]:
mean_hours_by_country_salary = data.pivot_table(index="native-country", columns="salary", values="hours-per-week", aggfunc="mean")
mean_hours_by_country_salary.rename(columns={">50K": "більше 50K", "<=50K": "не більше 50K"}).round(2)

salary,не більше 50K,більше 50K
native-country,,
?,40.16,45.55
Cambodia,41.42,40.00
Canada,37.91,45.64
China,37.38,38.90
Columbia,38.68,50.00
Cuba,37.99,42.44
Dominican-Republic,42.34,47.00
Ecuador,38.04,48.75
El-Salvador,36.03,45.00
